# Stage 3: Face Quality Assessment & Visual Quality Breakdown
Demonstrating **Resolution, Motion Blur, and Pose Symmetry Evaluation** with rich visual overlays for **DynaCampus-FR**.

In [ ]:
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Add project root to sys.path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.quality import FaceQualityEstimator

print("✅ Successfully imported FaceQualityEstimator!")

## 1. Create Sharp vs Blurry & Low-Res CCTV Face Probe Samples
Generating synthetic face crops representing high-quality CCTV frames vs heavily degraded probe tracks.

In [ ]:
estimator = FaceQualityEstimator()

# High Quality Sharp Face
sharp_face = np.full((112, 112, 3), 180, dtype=np.uint8)
cv2.circle(sharp_face, (35, 45), 10, (30, 30, 30), -1)
cv2.circle(sharp_face, (77, 45), 10, (30, 30, 30), -1)
cv2.ellipse(sharp_face, (56, 75), (20, 10), 0, 0, 180, (40, 40, 180), 3)
cv2.line(sharp_face, (10, 10), (102, 102), (255, 255, 255), 2)

# Low Quality Blurry Face
blurry_face = cv2.GaussianBlur(sharp_face, (19, 19), 7.0)

# Frontal Landmarks
landmarks_frontal = np.array([
    [35.0, 45.0], [77.0, 45.0], [56.0, 60.0], [38.0, 80.0], [74.0, 80.0]
], dtype=np.float32)

# Compute Quality Scores
q_sharp = estimator.estimate_quality(sharp_face, bbox=[0, 0, 112, 112], landmarks=landmarks_frontal)
q_blur = estimator.estimate_quality(blurry_face, bbox=[0, 0, 40, 40], landmarks=landmarks_frontal)

print("Sharp Face Metrics:", q_sharp)
print("Blurry Face Metrics:", q_blur)

## 2. Visual Annotated Quality Overlays
Overlaying visual quality badges and color-coded status banners directly on face crops.

In [ ]:
vis_sharp = estimator.visualize_quality(sharp_face, q_sharp)
vis_blur = estimator.visualize_quality(blurry_face, q_blur)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(cv2.cvtColor(vis_sharp, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Sharp Face Crop\nQuality Score: {q_sharp['overall_quality']:.2f}")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(vis_blur, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Degraded Probe Crop\nQuality Score: {q_blur['overall_quality']:.2f}")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 3. Multi-Metric Quality Breakdown Bar Chart
Comparing Resolution, Sharpness, Pose, and Composite Quality Scores.

In [ ]:
categories = ["Sharpness (Blur)", "Resolution", "Pose Symmetry", "Overall Quality"]
scores_sharp = [q_sharp["blur_score"], q_sharp["resolution_score"], q_sharp["pose_score"], q_sharp["overall_quality"]]
scores_blur = [q_blur["blur_score"], q_blur["resolution_score"], q_blur["pose_score"], q_blur["overall_quality"]]

x = np.arange(len(categories))
width = 0.35

plt.figure(figsize=(9, 5))
rects1 = plt.bar(x - width/2, scores_sharp, width, label="High Quality (Clear)", color="#2ecc71")
rects2 = plt.bar(x + width/2, scores_blur, width, label="Low Quality (Degraded)", color="#e74c3c")

plt.ylabel("Quality Score (0.0 - 1.0)")
plt.title("Face Quality Score Breakdown Comparison")
plt.xticks(x, categories)
plt.ylim(0, 1.15)
plt.axhline(0.70, color="green", linestyle="--", alpha=0.6, label="High Quality Threshold (0.70)")
plt.axhline(0.40, color="orange", linestyle="--", alpha=0.6, label="Acceptable Threshold (0.40)")
plt.legend(loc="upper right")

for rect in rects1 + rects2:
    height = rect.get_height()
    plt.annotate(f"{height:.2f}",
                xy=(rect.get_x() + rect.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center", va="bottom")

plt.tight_layout()
plt.show()